# Chapter 8: Adversarial Test Discovery for Agentic Systems

**The question.** An AI agent that writes validation reports can be pushed, by accident or on purpose, into reporting a wrong number. Before trusting it, attack it on purpose and see what gets through. Chapter 8 does this by design: list the attacks, list the situations they are tried in, try every attack in every situation and write down in advance what should happen.

**What this notebook attacks.** Not the agent itself, but the checker the agent relies on: the claim verifier of Chapter 7. A report makes a claim such as "the contrast of this region is 0.058326", and the verifier looks up the recorded value and returns a verdict. The failure to hunt for is a *false accept*: the verifier says Supported to a claim that is wrong.

**The steps.**

1. Pick two true facts to attack: two measurements of the same region.
2. Build five versions of a claim about each fact, one correct and four altered, each with the verdict it should get.
3. Run all 10 claims through the verifier and compare its verdicts with the expected ones.
4. State what this test covers and what it does not.

**Result.** All 10 verdicts matched what was expected, and no altered claim was accepted. This checks the verifier's rules on hand-made inputs; it does not measure how an AI agent behaves.

**Map of this notebook.**

| Step | Question | Where to look |
|---|---|---|
| 1 | Which true facts are attacked? | `value` column |
| 2 | What are the attacks? | `attack`, `expected` and `value_type` columns |
| 3 | Did the verifier give the expected verdicts? | the verdict grid |
| 4 | Did any wrong claim get through? | `false_accepts` column |

**Terms.**

- *Claim*: an address in the evidence store, such as `region:confirmation/reflection_73101/contrast`, plus the number a report says is stored there.
- *Verdicts*: **Supported** (the number matches the record), **Contradicted** (the record exists but the number differs), **Unscorable** (the claim cannot be checked, for example because the number is written as text) and **Absent** (the store has no record at that address).
- *Attack* (the book's *Inner Array*): one way of altering a claim. *Reference case* (the book's *Outer Array*): the true fact an attack is applied to.
- *False accept*: an altered claim that gets Supported.

**Data.** The evidence store is `book/evidence/parallel_aml_20260921/store.json`; its regional records come from the run used in Chapters 2, 4 and 5. The customers are simulated. No language model is called. Run the cells in order; the `assert` lines check that every verdict equals its expected verdict.

## Setup and source checks

Loads the helper code and the saved evidence and checks that the files have not changed. The verifier comes from `book/parallel_evidence/build.py`. Select the kernel of the environment where `aiv` is installed.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT).as_posix())
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Step 1: Pick the true facts to attack

The two facts are two different measurements of the same frozen region: the seed-73101 region that Chapter 5's two-cluster rule found, measured on the 200,000 confirmation customers. One is its average Brier loss, the other its contrast. The table shows each fact's stored value and where it comes from. Chapter 8 calls these facts *reference cases*, the Outer Array of the design (Section 8.2).

In [2]:
from book.parallel_evidence.build import OUT, verify
store=read(OUT/"store.json")
outer=["region:confirmation/reflection_73101/brier","region:confirmation/reflection_73101/contrast"]
display(pd.DataFrame([dict(case=t,value=store["facts"][t]["value"],source=store["facts"][t]["source"],
                           pointer=store["facts"][t]["pointer"]) for t in outer]))

,case,value,source,pointer
0,region:confirmation/reflection_73101/brier,0.072851,book/evidence/aml_medoid_loop_leaf_output_2026...,/confirmation/reflection_73101/brier
1,region:confirmation/reflection_73101/contrast,0.058326,book/evidence/aml_medoid_loop_leaf_output_2026...,/confirmation/reflection_73101/contrast


**Read the output.** **In one line: two true numbers about the same region, stored at addresses that differ only in the last word.**

**Columns.** `case`: the fact's address in the evidence store. `value`: the recorded number (Brier loss 0.072851, contrast 0.058326). `source`: the results file it was read from, cut off in the display. `pointer`: where in that file the number sits.

**Notice.** Because the two addresses differ only in `brier` against `contrast`, this pair tests whether the verifier ties a number to the right measurement or accepts any number it has on record.

## Step 2: Build the attacks

Each attack alters a correct claim in one way. Its expected verdict is written down before anything is checked, so the test cannot be adjusted after seeing the results. The table shows the five claims built from the contrast fact; the book calls the attacks the Inner Array (Section 8.2).

- `faithful`: the correct claim; expected **Supported**.
- `wrong_value`: the right address with 0.1 added to the number; expected **Contradicted**.
- `wrong_metric`: the contrast's number placed at the Brier loss address; expected **Contradicted**.
- `numeric_string`: the right address and number, but the number written as text; expected **Unscorable**.
- `unsupported_repair`: the number placed at the address of a result that was never measured (`predictor_repair_gain`); expected **Absent**.

In [3]:
inner=["faithful","wrong_value","wrong_metric","numeric_string","unsupported_repair"]
expected={"faithful":"Supported","wrong_value":"Contradicted","wrong_metric":"Contradicted",
          "numeric_string":"Unscorable","unsupported_repair":"Absent"}
def mutate(target,attack):
    value=round(store["facts"][target]["value"],6)
    id_,claimed=target,value
    if attack=="wrong_value":claimed=value+.1
    elif attack=="wrong_metric":id_=target.rsplit("/",1)[0]+("/contrast" if target.endswith("/brier") else "/brier")
    elif attack=="numeric_string":claimed=str(value)
    elif attack=="unsupported_repair":id_="region:experiment/predictor_repair_gain"
    return {"id":id_,"value":claimed}
display(pd.DataFrame([dict(attack=a,expected=expected[a],**(c:=mutate(outer[1],a)),value_type=type(c["value"]).__name__) for a in inner]))

,attack,expected,id,value,value_type
0,faithful,Supported,region:confirmation/reflection_73101/contrast,0.058326,float
1,wrong_value,Contradicted,region:confirmation/reflection_73101/contrast,0.158326,float
2,wrong_metric,Contradicted,region:confirmation/reflection_73101/brier,0.058326,float
3,numeric_string,Unscorable,region:confirmation/reflection_73101/contrast,0.058326,str
4,unsupported_repair,Absent,region:experiment/predictor_repair_gain,0.058326,float


**Read the output.** **In one line: each row is one claim exactly as the verifier will receive it.**

**Columns.** `attack`: which alteration. `expected`: the verdict declared in advance. `id`: the address the claim points to. `value`: the claimed number. `value_type`: `float` for a number, `str` for text.

**Notice.** `wrong_metric` is the sneakiest attack: 0.058326 is a real, correct number, only reported as the wrong measurement. `numeric_string` differs from the correct claim only in type; a verifier that quietly converted text to numbers would accept it.

## Step 3: Run every attack on every fact

The cell tries all 5 attacks on both facts, 10 claims in all. It checks each verdict against the expected one and marks a false accept whenever an altered claim is Supported. The grid shows the verdicts, one row per attack and one column per fact.

In [4]:
rows=[]
for target in outer:
    for attack in inner:
        verdict=verify(store,mutate(target,attack))["verdict"]
        assert verdict==expected[attack]
        rows.append(dict(case=target,attack=attack,verdict=verdict,false_accept=attack!="faithful" and verdict=="Supported"))
design=pd.DataFrame(rows)
display(design.pivot(index="attack",columns="case",values="verdict"))

case,region:confirmation/reflection_73101/brier,region:confirmation/reflection_73101/contrast
attack,,
faithful,Supported,Supported
numeric_string,Unscorable,Unscorable
unsupported_repair,Absent,Absent
wrong_metric,Contradicted,Contradicted
wrong_value,Contradicted,Contradicted


**Read the output.** **In one line: every claim got its expected verdict, on both facts.**

**Columns.** Rows are the attacks, sorted alphabetically. The two columns are the two facts, Brier loss and contrast. Each entry is the verifier's verdict.

**Notice.** The two columns are identical, so the verifier behaves the same whichever measurement is attacked, and no altered claim is Supported. The attacks were hand-made and handed to the verifier as ready address-and-number pairs. A real agent first has to pull such claims out of its own text, and that step, which is not tested here, is where substitutions like `wrong_metric` would happen.

## Step 4: State what was and was not tested

A test report should say what was tested and what was not. The table counts, for each attack, how many claims were tried and how many were wrongly accepted. The printed lines state what this notebook did not test.

In [5]:
display(design.groupby("attack").agg(cases=("verdict","size"),false_accepts=("false_accept","sum")))
print("Live regional agent calls: 0")
print("No conversation-separation or prompt-resistance estimate is supplied.")

,cases,false_accepts
attack,,
faithful,2,0
numeric_string,2,0
unsupported_repair,2,0
wrong_metric,2,0
wrong_value,2,0


Live regional agent calls: 0
No conversation-separation or prompt-resistance estimate is supplied.


**Read the output.** **In one line: zero false accepts, but only for hand-made claims fed straight to the verifier.**

**Columns.** `cases`: claims tried with this attack, one per fact. `false_accepts`: altered claims the verifier wrongly Supported.

**Notice.** This shows that the verifier follows its rules on these inputs. It does not show how often a language model would make such mistakes, how the verifier handles claims pulled from free text or what happens over a multi-turn conversation. The live tests with real model calls, 70 calls crossing four prompt rules with two source-access conditions, are in `regional_live/ch08_adversarial_prompts.ipynb`. The chapter's multi-turn conversation experiment is in the historical companion `notebooks/ch07_adversarial_prompts.ipynb`.

## What we learned

We attacked the claim checker that a reporting agent relies on. Four kinds of wrong claim (a wrong number, a right number under the wrong measurement, a number written as text and a claim about a result that does not exist) each got the verdict written down in advance, on both facts, and none was accepted.

The design is the lesson: list the attacks, list the situations, try every combination and fix the expected outcome first. What was tested is the checker's rules, not an AI agent. A live test would add the model's own answers, the step that pulls claims out of its text and a scorer whose own mistakes must be measured. Chapter 9 returns to the frozen regions and asks what their measurements establish as findings.


## What to try

1. **Attack with the wrong study.** Add an attack that moves the claim to `moe:confirmation/llm_compromise/worst_logloss`. Write down its expected verdict first, then add it to `expected` and `mutate` and rerun the grid.
2. **A third fact.** Add `region:confirmation/reflection_73101/share` as a third fact. Which attacks still make sense for a share, and why?
3. **A near miss.** Add an attack that changes the number by 0.0000004. Predict its verdict from the tolerance in Chapter 7. Is the result a false accept?
4. **A live test.** Write a user prompt that tries to make an agent report the contrast as the region's Brier loss. What would a live study need to record to measure how often the agent falls for it, and to tell the agent's mistakes from the scorer's?
